## Notebook Stage

Realiza o download e persistência no Volume dos arquivos utilizados no projeto. 
<p>Os dados de dados da ANP podem ser apresentados nos formatos .csv e .zip. Caso o Arquivo baixado esteja em formato .zip, este é mantido na pasta '..ANP/PRECOS/ORIG' e seu conteúdo é extraído para a pasta '../ANP/PRECOS/CSV', em que todos arquivos de dados da ANP são gravados</p>
<p>Os arquivos de vendas de Etanol e Gasolina são baixados já em sua forma .csv e armazenados em "../ANP/VENDAS</p>
<p>O arquivo de municipios do IBGE é salvo em "../IBGE", e mantido em seu formato .json (NoSQL)</p>
<br>Arquivos são armazenados no Volume "00_raw".

In [0]:
# Imports

import shutil
from pathlib import Path
from zipfile import ZipFile

import requests

##### Definição e criação dos caminhos

In [0]:
CATALOGO = 'ANP_Combustiveis'
SCHEMA   = '00_raw'
Volume   = 'data'

volumePath = Path(f'/Volumes/{CATALOGO}/{SCHEMA}/{Volume}')

paths = {
    'PRECOS_ORIGINAIS': Path(volumePath / 'ANP' / 'PRECOS' / 'ORIG'),
    'PRECOS_CSV':       Path(volumePath / 'ANP' / 'PRECOS' / 'CSV'),
    'VENDAS':           Path(volumePath / 'ANP' / 'VENDAS'),
    'IBGE':             Path(volumePath / 'IBGE')
}

for _, path in paths.items():
    if not path.exists():
        path.mkdir(parents=True, exist_ok=True)

##### Definição dos URLs para obtenção dos arquivos

In [0]:
URL_ANP = "https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/shpc/dsas/ca"

# Arquivos a serem baixados - (Poderia ser otimizado através de WebScrapping- BeautifulSoup)
ANPfileNames = [
    # Arquivos *.csv [Pré 2022]
    "ca-2017-01.csv",
    "ca-2017-02.csv",
    "ca-2019-01.csv",
    "ca-2019-02.csv",
    "ca-2020-01.csv",
    "ca-2020-02.csv",
    "ca-2021-01.csv",
    "ca-2021-02.csv",
    # Arquivos *.zip [2022 - atual]
    "ca-2022-02.zip",
    "ca-2023-01.zip",
    "ca-2023-02.zip",
    "ca-2024-01.zip",
    "ca-2024-02.zip",
    "ca-2025-01.zip",
    "ca-2025-02.zip",
]

# URLs de vendas de combustíveis [ANP] - Gasolina & Etanol 
URL_Gasolina = (
    "https://www.gov.br/anp/pt-br/centrais-de-conteudo/"
    "dados-abertos/arquivos/vdpb/vaehdpm/gasolina-c/"
    "vendas-anuais-de-gasolina-c-por-municipio.csv"
)

URL_Etanol = (
    "https://www.gov.br/anp/pt-br/centrais-de-conteudo/"
    "dados-abertos/arquivos/vdpb/vaehdpm/etanol-hidratado/"
    "vendas-anuais-de-etanol-hidratado-por-municipio.csv"
)

url_IBGE_Municipios = "https://servicodados.ibge.gov.br/api/v1/localidades/municipios?orderBy=nome"

##### Baixar arquivos

In [0]:
# DEFINIÇÃO DE FUNÇÕES AUXILIARES

# Baixa os arquivos a partir de uma URL
def downloadFile(url: str, filePath: Path):
    if filePath.exists():
        print(f"[Skip-Download] Arquivo já existente: {filePath.name}"); return

    # Executa requisição
    with requests.get(url, headers={ "User-Agent": "Mozilla/5.0"}, stream=True, timeout=180) as r:
        r.raise_for_status()
        totalBytes = int(r.headers.get("Content-Length", 0))
        downloadedBytes = 0

        # Escreve o conteúdo no arquivo (binário)
        with filePath.open("wb") as outFile:
            for i, chunk in enumerate(r.iter_content(chunk_size=1024 * 1024)):
                if not chunk:
                    continue
                
                outFile.write(chunk)

                # Imprime progresso 
                downloadedBytes += len(chunk)
                downloadedMb = downloadedBytes / 1024**2
                if totalBytes:
                    totalMb = totalBytes / 1024**2
                    progress = downloadedBytes / totalBytes * 100

                    print(f"\r[Downloading] {filePath.name}: {progress:.1f}% ({downloadedMb:.1f}/{totalMb:.1f} MB)", 
                          end="",flush=True,)
                else:
                    print(f"\r[Downloading] {filePath.name}: {downloadedMb:.1f} MB", end="", flush=True)

# Extrai o conteúdo de *.csv um arquivo *.zip
def csvExtract(zipPath: Path, csvPath: Path):
    if csvPath.exists():
        print(f'[Skip-Extract] Arquivos .csv já extraídos de \'{zipPath}\''); return
    
    with ZipFile(zipPath) as fileZip:
        # Lista todos arquivos *.csv
        csvFiles = []
        for file in fileZip.namelist():
            if file.lower().endswith(".csv"):
                csvFiles.append(file)
        
        # Copia o conteúdo do arquivo *.csv extraído para o arquivo de destino
        with (fileZip.open(csvFiles[0]) as sourcePath, csvPath.open("wb") as targetPath):
            shutil.copyfileobj(sourcePath, targetPath)

In [0]:
## Obtém os arquivos de preços de combustíveis [ANP]

for fileName in ANPfileNames:
    url = f"{URL_ANP}/{fileName}"

    if fileName.endswith(".zip"):
        zipPath = Path(paths['PRECOS_ORIGINAIS'] / fileName)
        csvPath = Path(paths['PRECOS_CSV'] / fileName.replace(".zip", ".csv"))

        downloadFile(url, zipPath)
        csvExtract(zipPath, csvPath)
    else:
        csvPath = Path(paths['PRECOS_CSV'] / fileName)

        downloadFile(url, csvPath)

In [0]:

csvPath_gasolinaComum   = Path(paths['VENDAS'] / 'vendas-gasolina-c.csv')
csvPath_etanolHidratado = Path(paths['VENDAS'] / 'vendas-etanol-hidratado.csv')
jsonPath_municipiosIBGE = Path(paths['IBGE'] / 'municipios.json')

# Download dos arquivos de vendas de combustíveis [ANP]
downloadFile(URL_Gasolina, csvPath_gasolinaComum)
downloadFile(URL_Etanol, csvPath_etanolHidratado)

# Download do arquivos de municípios [IBGE]
downloadFile(url_IBGE_Municipios, paths['IBGE'] / "municipios.json")

In [0]:
for arquivo in sorted(CAMINHO_VOLUME.rglob("*")):
    if arquivo.is_file():
        tamanho_mb = (
            arquivo.stat().st_size / (1024 * 1024)
        )

        print(
            f"{arquivo.relative_to(CAMINHO_VOLUME)} "
            f"— {tamanho_mb:.2f} MB"
        )